# 🎓 AI Coding Tutor Using Tiny RAG


Let us make a simple AI coding tutor using Tiny RAG. First, we set-up the model. In the code below, we install and import the necessary tools.



##🤖 Part 1: Setting Up the Model

In [ ]:
!pip install -q sentence-transformers faiss-cpu google-generativeai groq

from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import google.generativeai as genai
from groq import Groq
import os


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.1 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)



Next we have to provide our API key for communicating with Groq servers.

🚨 Please put here your own API key, which you can get at: https://console.groq.com/keys

In [ ]:
client = Groq(api_key="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")

(Below is a command for displaying all available *Gemini* models.)

In [ ]:
#for m in genai.list_models():
#   if "generateContent" in m.supported_generation_methods:
#       print(m.name)

##📚 Part 2: Building Tiny RAG Knowledge Base

Let us come to Tiny RAG. Below is a simple Python dictionary containing coding teaching material.

In [ ]:
rag_material = [
    {
        "topic": "Python Lists",
        "content": "A Python list is an ordered, mutable collection of items. "
                   "Example of a Python list: my_list = [1, 2, 3]. "
                   "Lists can hold any type: strings, numbers, even other lists."
    },
    {
        "topic": "Python Functions",
        "content": "A function in Python is defined using the def keyword. "
                   "Example: def add(a, b): return a + b. "
                   "Functions take inputs (parameters) and return outputs."
    },
    {
        "topic": "Python Loops",
        "content": "A loop allows you to repeat actions. "
                   "Example: for i in range(5): print(i). "
                   "Common mistake: forgetting indentation in Python causes errors."
    },
    {
        "topic": "Python Strings",
        "content": "Strings in Python are sequences of characters. "
                   "Example: text = 'hello'. "
                   "Strings are immutable — you cannot change individual characters in place."
    },
]


Now let's do an **embedding** 👉 this ascribes **a vector** (a list of numbers) to each document. In this way, we can search for documents according to their **meaning**, not only words that they have in common.

In [ ]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
material_embeddings = embedder.encode(rag_material)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ValueError: Multimodal dict input contains unrecognized modality keys: {'content', 'topic'}. Expected keys from: {'audio', 'video', 'image', 'text'}

By running the command *.shape* on our RAG Python dictionary, we receive two numbers - first is the number of documents we have, second is the number of dimensions of the vector assigned to each document.

In [ ]:
material_embeddings.shape

Now we **index** our RAG material using **FAISS**. This index serves a similiar purpose to an index in a book - it makes the searching for content faster.

In [ ]:
dimension = material_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(np.array(material_embeddings))

Next we define the *retrieve* function. We search **semantically** via vectors.

In [ ]:
def retrieve(query, k=1):
    query_embedding = embedder.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)
    result = [rag_material[i] for i in indices[0]]
    prompt = f"""You have a Python Dictionary entry: {result}.

        Optimize it for plain text display. Capitalize letter where suitable.

        The final result should look like a text, not a dictionary entry.

        Make the indentation, punctuation and paragraphs consistent.

        Make sure you process the whole dictionary entry including the key and the value.

        This prompt is only for retrieval. Do not add anything which you say yourself, just the data.

        Make the output so that it can be directly pasted."""

    llm_output = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
        )
    response = llm_output.choices[0].message.content
    return response

Let's see if our retrieve function works properly.

In [ ]:
print(retrieve("What is a loop?"))

Looks good. Let's see if we can retrieve our RAG entry using words not in the entry, thereby testing our semantic retrieval. Let's say "combination of letters" instead of "sequences of characters".

In [ ]:
print(retrieve("How do you store a combination of letters in Python?"))

Now we move on to making a prompt for the tutor itself.

In [ ]:
def build_prompt(context, question):
    return f"""
You are a coding tutor.

Use the context to answer the question when appropriate, sometimes it does not make sense to use it.

Context:
{context}

Question:
{question}

Instructions:
- Adjust your response visually for plain text -- things like ** and `` do not display properly (although emojis work fine)
- Add emojis for beauty, nut not too much
- Explain simply (beginner-friendly)
- Give a real-life coding example of how it can be used
- Mention common mistakes
- Ask a follow-up question
- If there is sufficient context, mention that also and say where you got it from - tiny rag.

"""

In [ ]:
def ask_tutor(question):
    context = retrieve(question, k=1)

    prompt = build_prompt(context, question)

    llm_output = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
        )

    response = llm_output.choices[0].message.content

    return response


Let us test our tutor.

In [ ]:
print(ask_tutor("What is a Python list?"))

Now something harder.

In [ ]:
SQL_query = """
SELECT
    customer_id,
    amount,
    SUM(amount) OVER (PARTITION BY customer_id) AS total_per_customer
FROM orders;
"""

print(ask_tutor("Explain this query to me: " + SQL_query))

Can our tutor solve a Python bug?

In [ ]:
error = """
age = input("Enter your age: ")
print("Next year you will be", age + 1)"""

print(ask_tutor("Why doesn't this code work? - " + error))

##🎯 Part 3: Exercises

Make a team of **2-3 people**. Complete the exercises below. The fastest team wins 🏆

📌 **1. Task:** Put another dictionary entrance into our Tiny RAG Knowledge Base. Demonstrate the ability to search there *semantically* - searching by words not found in the new RAG entrance itself.

📌 **2. Task:** Edit this Jupyter Notebook, so that the **ask_tutor** funcionality uses a different LLM model than Groq.

(Recommendations and tips: Use Gemini. You will need another (Gemini) API key. Some Gemini agents are free for a limited amounts of API calls. Feel free to use AI for help.)

You can send the result to petr.kalina.jr@gmail.com
